<a href="https://colab.research.google.com/github/gabrielsanchez/sussex-thesis/blob/main/Leakage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Leakage tracker: non-Pauli noise alongside Stim

Leakage to non-computational states is the noise channel that most distinguishes neutral-atom hardware from superconducting qubits, and it's fundamentally non-Pauli (Stim cannot represent it natively). This notebook builds a leakage simulator that runs alongside Stim's FlipSimulator, tracking a per-qubit per-shot leakage state and injecting the appropriate Pauli noise as a consequence.

Mechanism modeled:

    1. Leakage seeding. Each two-qubit gate, each participating atom has probability p_leak_per_2q_gate of leaking to a non-computational state. (In real hardware this comes from spontaneous decay out of the Rydberg state during the gate.)
    
    2. Partner-qubit corruption. When a leaked atom participates in a 2q gate, the partner is not entangled correctly and ends up effectively maximally mixed. This is modelled as a uniform Pauli (X/Y/Z) applied to the partner with total probability partner_depol_strength.
    
    3. Measurement randomization. A leaked ancilla measurement returns an outcome statistically uncorrelated with the parity it was supposed to detect. This is modelled by flipping the measurement outcome with 50% probability whenever the measured qubit is leaked.
    
    4. Reset cures leakage. A reset operation returns a leaked atom to the computational subspace with probability p_reset_cures. (This is the main reason MR — measure then reset — keeps ancilla leakage from accumulating across rounds.)
    
    5. Decay between rounds. Each round, leaked atoms return to computational with probability p_decay_per_round, applying a random Pauli on return.



In [ ]:
!pip install stim pymatching

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.2/626.2 kB 43.9 MB/s eta 0:00:00


In [ ]:
import json
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import stim
import pymatching
import matplotlib.pyplot as plt

print("stim", stim.__version__)
print("pymatching", pymatching.__version__)

stim 1.16.0
pymatching 2.4.0


## Noise Model and Circuit Builder

This is the same model as defined in the previous notebooks.

In [ ]:
@dataclass
class NeutralAtomNoise:
    p_depol1: float = 1e-3
    p_depol2: float = 5e-3
    p_meas: float = 5e-3
    p_reset: float = 1e-3
    p_correlated_zz: float = 1e-3
    p_erasure: float = 2e-3
    erasure_depol_strength: float = 0.75


def build_syndrome_circuit(distance, rounds, noise):
    template = stim.Circuit.generated(
        code_task="surface_code:rotated_memory_z",
        distance=distance, rounds=rounds,
    )

    out = stim.Circuit()
    for inst in template:
        if isinstance(inst, stim.CircuitRepeatBlock):
            inner = stim.Circuit()
            for sub in inst.body_copy():
                _emit_syndrome(sub, inner, noise)
            out.append(stim.CircuitRepeatBlock(inst.repeat_count, inner))
        else:
            _emit_syndrome(inst, out, noise)
    return out


def _emit_syndrome(inst, out, n):
    name = inst.name
    targets = inst.targets_copy()
    if name in ("M", "MX", "MY", "MZ"):
        qbits = [t.value for t in targets if t.is_qubit_target]
        if qbits and n.p_meas > 0:
            out.append("X_ERROR", qbits, n.p_meas)
        out.append(inst); return
    if name == "MR":
        qbits = [t.value for t in targets if t.is_qubit_target]
        if qbits and n.p_meas > 0:
            out.append("X_ERROR", qbits, n.p_meas)
        out.append(inst)
        if qbits and n.p_reset > 0:
            out.append("X_ERROR", qbits, n.p_reset)
        return
    if name in ("R", "RX", "RY", "RZ"):
        out.append(inst)
        qbits = [t.value for t in targets if t.is_qubit_target]
        if qbits and n.p_reset > 0:
            out.append("X_ERROR", qbits, n.p_reset)
        return
    if name in ("CX", "CZ", "CY"):
        out.append(inst)
        qbits = [t.value for t in targets if t.is_qubit_target]
        if n.p_depol2 > 0:
            out.append("DEPOLARIZE2", qbits, n.p_depol2)
        if n.p_erasure > 0 and n.erasure_depol_strength > 0:
            out.append("DEPOLARIZE1", qbits, n.p_erasure * n.erasure_depol_strength)
        if n.p_correlated_zz > 0:
            for i in range(0, len(qbits), 2):
                out.append("CORRELATED_ERROR",
                    [stim.target_z(qbits[i]), stim.target_z(qbits[i + 1])],
                    n.p_correlated_zz)
        return
    if name in ("H", "S", "S_DAG", "X", "Y", "Z",
                "SQRT_X", "SQRT_Y", "SQRT_X_DAG", "SQRT_Y_DAG",
                "C_XYZ", "C_ZYX"):
        out.append(inst)
        qbits = [t.value for t in targets if t.is_qubit_target]
        if qbits and n.p_depol1 > 0:
            out.append("DEPOLARIZE1", qbits, n.p_depol1)
        return
    out.append(inst)


## Leakage Params

In [ ]:
@dataclass
class LeakageParams:
    """Parameters for the leakage tracker.

    Defaults aim for the regime of recent neutral-atom experiments:
    sub-percent leakage per 2q gate, mostly cured by reset, modest
    decay between rounds.
    """
    p_leak_per_2q_gate: float = 1e-3      # per atom per 2q gate
    p_decay_per_round: float = 0.10       # leaked -> computational, random Pauli
    p_reset_cures: float = 0.95           # reset returns to computational
    partner_depol_strength: float = 0.75  # uniform Pauli on partner of leaked atom

## Leakage-aware sampler

In [ ]:
def sample_with_leakage(
    circuit: stim.Circuit,
    num_shots: int,
    leakage: LeakageParams,
    seed: int = 0,
):
    """Sample syndromes while tracking per-shot leakage state.

    Returns:
        det_flips: (num_shots, num_detectors) uint8
        obs_flips: (num_shots, num_observables) uint8
        leakage_record: (num_shots, num_qubits) int — cumulative
            qubit-rounds spent leaked.
        leak_per_round: (num_shots, num_rounds, num_qubits) bool —
            was qubit q leaked at the end of round r (after decay
            but before the next round's gates).
    """
    rng = np.random.default_rng(seed)
    n_qubits = circuit.num_qubits

    sim = stim.FlipSimulator(
        batch_size=num_shots,
        num_qubits=n_qubits,
        disable_stabilizer_randomization=True,
        seed=int(rng.integers(2**31)),
    )

    leaked = np.zeros((n_qubits, num_shots), dtype=bool)
    leakage_record = np.zeros((num_shots, n_qubits), dtype=np.int32)
    leak_per_round: list[np.ndarray] = []

    for inst in circuit.flattened():
        name = inst.name
        targets = inst.targets_copy()

        # ----- Two-qubit Cliffords -----
        if name in ("CX", "CZ", "CY"):
            qbits = [t.value for t in targets if t.is_qubit_target]

            # Inject partner-corruption noise BEFORE the gate
            px = np.zeros((n_qubits, num_shots), dtype=bool)
            py = np.zeros((n_qubits, num_shots), dtype=bool)
            pz = np.zeros((n_qubits, num_shots), dtype=bool)
            for i in range(0, len(qbits), 2):
                q0, q1 = qbits[i], qbits[i + 1]
                # q0 leaked, q1 not -> noise on q1
                _sample_uniform_pauli(
                    rng, leaked[q0] & ~leaked[q1],
                    leakage.partner_depol_strength, q1, px, py, pz,
                )
                # q1 leaked, q0 not -> noise on q0
                _sample_uniform_pauli(
                    rng, leaked[q1] & ~leaked[q0],
                    leakage.partner_depol_strength, q0, px, py, pz,
                )
            if px.any(): sim.broadcast_pauli_errors(pauli='X', mask=px)
            if py.any(): sim.broadcast_pauli_errors(pauli='Y', mask=py)
            if pz.any(): sim.broadcast_pauli_errors(pauli='Z', mask=pz)

            # Apply the gate
            sim.do(inst)

            # Seed new leakage events AFTER the gate
            for i in range(0, len(qbits), 2):
                q0, q1 = qbits[i], qbits[i + 1]
                new0 = (rng.random(num_shots) < leakage.p_leak_per_2q_gate) & ~leaked[q0]
                new1 = (rng.random(num_shots) < leakage.p_leak_per_2q_gate) & ~leaked[q1]
                leaked[q0] |= new0
                leaked[q1] |= new1
            continue

        # ----- Measurements (incl. MR for ancillas, which is a round boundary) -----
        if name in ("M", "MR", "MX", "MY", "MZ"):
            qbits = [t.value for t in targets if t.is_qubit_target]

            # Leaked qubit -> randomize measurement outcome (50/50 flip)
            flip_mask = np.zeros((n_qubits, num_shots), dtype=bool)
            for q in qbits:
                flip_mask[q] = leaked[q] & (rng.random(num_shots) < 0.5)
            if flip_mask.any():
                # X flips Z-basis measurements (M, MR, MZ); Z flips X-basis
                pauli_for_flip = 'X' if name in ("M", "MR", "MZ") else (
                    'Z' if name == "MX" else 'X'
                )
                sim.broadcast_pauli_errors(pauli=pauli_for_flip, mask=flip_mask)

            sim.do(inst)

            if name == "MR":
                # Reset on the ancillas cures leakage probabilistically
                for q in qbits:
                    cured = (rng.random(num_shots) < leakage.p_reset_cures) & leaked[q]
                    leaked[q] &= ~cured

                # End-of-round leakage decay (on all qubits, not just ancillas)
                decay = (rng.random((n_qubits, num_shots)) < leakage.p_decay_per_round) & leaked
                dx = decay & (rng.random((n_qubits, num_shots)) < 0.5)
                dz = decay & (rng.random((n_qubits, num_shots)) < 0.5)
                if dx.any(): sim.broadcast_pauli_errors(pauli='X', mask=dx)
                if dz.any(): sim.broadcast_pauli_errors(pauli='Z', mask=dz)
                leaked &= ~decay

                leak_per_round.append(leaked.T.copy())
                leakage_record += leaked.T.astype(np.int32)
            continue

        # ----- Resets -----
        if name in ("R", "RX", "RY", "RZ"):
            sim.do(inst)
            qbits = [t.value for t in targets if t.is_qubit_target]
            for q in qbits:
                cured = (rng.random(num_shots) < leakage.p_reset_cures) & leaked[q]
                leaked[q] &= ~cured
            continue

        # ----- Everything else: pass through -----
        sim.do(inst)

    # Final cumulative record after the last instruction
    leakage_record += leaked.T.astype(np.int32)

    det_flips = sim.get_detector_flips(bit_packed=False).T.astype(np.uint8)
    obs_flips = sim.get_observable_flips(bit_packed=False).T.astype(np.uint8)
    leak_arr = (np.stack(leak_per_round, axis=1)
                if leak_per_round
                else np.zeros((num_shots, 0, n_qubits), dtype=bool))
    return det_flips, obs_flips, leakage_record, leak_arr


def _sample_uniform_pauli(rng, affected_mask, prob, qubit, xm, ym, zm):
    """For shots in affected_mask, apply a uniform random Pauli to `qubit`
    with probability `prob`. Updates the per-Pauli output masks in place."""
    fires = affected_mask & (rng.random(affected_mask.shape[0]) < prob)
    if not fires.any():
        return
    choice = rng.integers(0, 3, size=fires.shape[0])
    xm[qubit] |= fires & (choice == 0)
    ym[qubit] |= fires & (choice == 1)
    zm[qubit] |= fires & (choice == 2)

